# Huntsville MSA Labor Market – AI Exposure Analysis
Joins local employment data with occupation-level AI exposure scores and visualizes the exposure landscape.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

ROOT = Path('..').resolve()

## 1. Load data

In [2]:
labor = pd.read_csv(ROOT / 'Huntsville-MSA-Labor-Market-clean.csv')

# Strip leading $ and convert currency/numeric columns
for col in labor.columns[2:]:
    labor[col] = labor[col].astype(str).str.lstrip('$')
    labor[col] = pd.to_numeric(labor[col], errors='coerce')

print(f"Labor market rows: {len(labor)}")
labor.head(3)

Labor market rows: 571


,Occupational Title,SOC,Estimated Employment,Entry Hourly,Mean Hourly,Experienced Hourly,Entry Annual,Mean Annual,Experienced Annual
0,Total all occupations,00-0000,260190.0,15.04,34.78,44.65,31291.0,72342.0,92868.0
1,Management Occupations,11-0000,15730.0,35.32,68.44,85.00,73467.0,142351.0,176793.0
2,Top Executives,11-1000,4240.0,35.11,77.21,98.27,73016.0,160608.0,204404.0


In [3]:
exposure = pd.read_csv(ROOT / 'labor_market_impacts' / 'job_exposure.csv')
print(f"Exposure rows: {len(exposure)}")
exposure.head(3)

Exposure rows: 756


,occ_code,title,observed_exposure
0,11-1011,Chief Executives,0.0333
1,11-1021,General and Operations Managers,0.1378
2,11-1031,Legislators,0.0000


## 2. Join on SOC code

In [4]:
df = labor.merge(
    exposure[['occ_code', 'observed_exposure']],
    left_on='SOC',
    right_on='occ_code',
    how='inner'
).drop(columns='occ_code')

df = df.dropna(subset=['Estimated Employment', 'observed_exposure'])
df['Estimated Employment'] = df['Estimated Employment'].astype(int)

print(f"Matched occupations: {len(df)}")
df[['Occupational Title', 'SOC', 'Estimated Employment', 'observed_exposure']].head()

Matched occupations: 420


,Occupational Title,SOC,Estimated Employment,observed_exposure
0,Chief Executives,11-1011,90,0.0333
1,General and Operations Managers,11-1021,4090,0.1378
2,Legislators,11-1031,60,0.0000
3,Marketing Managers,11-2021,210,0.3195
4,Sales Managers,11-2022,610,0.0433


## 3. Aggregate: overall employment-weighted exposure

In [5]:
total_emp = df['Estimated Employment'].sum()
weighted_exposure = (df['observed_exposure'] * df['Estimated Employment']).sum() / total_emp

print(f"Total matched employment:         {total_emp:>10,}")
print(f"Employment-weighted avg exposure: {weighted_exposure:>10.3f}  ({weighted_exposure*100:.1f}%)")

Total matched employment:            229,930
Employment-weighted avg exposure:      0.135  (13.5%)


## 4. Derive broad occupation group from SOC prefix

In [6]:
SOC_GROUPS = {
    '11': 'Management',
    '13': 'Business & Finance',
    '15': 'Computer & Math',
    '17': 'Architecture & Engineering',
    '19': 'Life, Physical & Social Science',
    '21': 'Community & Social Service',
    '23': 'Legal',
    '25': 'Education & Library',
    '27': 'Arts, Design & Media',
    '29': 'Healthcare Practitioners',
    '31': 'Healthcare Support',
    '33': 'Protective Service',
    '35': 'Food Preparation',
    '37': 'Building & Grounds',
    '39': 'Personal Care & Service',
    '41': 'Sales',
    '43': 'Office & Admin Support',
    '45': 'Farming, Fishing & Forestry',
    '47': 'Construction & Extraction',
    '49': 'Installation, Maintenance & Repair',
    '51': 'Production',
    '53': 'Transportation & Material Moving',
}

df['Occupation Group'] = df['SOC'].str[:2].map(SOC_GROUPS).fillna('Other')

## 5. Visualisation A – Bubble chart: employment vs. exposure by occupation

In [7]:
fig = px.scatter(
    df,
    x='Mean Annual',
    y='observed_exposure',
    size='Estimated Employment',
    color='Occupation Group',
    hover_name='Occupational Title',
    hover_data={
        'SOC': True,
        'Estimated Employment': ':,',
        'observed_exposure': ':.3f',
        'Mean Annual': ':$,.0f',
    },
    size_max=60,
    title='Huntsville MSA – AI Exposure vs. Mean Annual Wage<br><sup>Bubble size = Estimated Employment</sup>',
    labels={
        'Mean Annual': 'Mean Annual Wage ($)',
        'observed_exposure': 'AI Exposure Score',
    },
)

fig.add_hline(
    y=weighted_exposure,
    line_dash='dash',
    line_color='black',
    annotation_text=f'Weighted avg: {weighted_exposure:.2f}',
    annotation_position='top right',
)

fig.update_layout(height=600, legend_title_text='Occupation Group')
fig.show()

## 6. Visualisation B – Treemap: employment weighted by exposure

In [8]:
fig2 = px.treemap(
    df,
    path=[px.Constant('All Occupations'), 'Occupation Group', 'Occupational Title'],
    values='Estimated Employment',
    color='observed_exposure',
    color_continuous_scale='RdYlGn_r',
    color_continuous_midpoint=0.25,
    hover_data={
        'observed_exposure': ':.3f',
        'Estimated Employment': ':,',
    },
    title='Huntsville MSA Labor Market – AI Exposure Treemap<br>'
          '<sup>Area = Estimated Employment | Color = AI Exposure (red = higher)</sup>',
)

fig2.update_layout(height=700, coloraxis_colorbar_title='Exposure')
fig2.show()

## 7. Visualisation C – Bar chart: exposure by occupation group (employment weighted)

In [9]:
group_stats = (
    df.groupby('Occupation Group')
    .apply(
        lambda g: pd.Series({
            'Weighted Exposure': (g['observed_exposure'] * g['Estimated Employment']).sum() / g['Estimated Employment'].sum(),
            'Total Employment': g['Estimated Employment'].sum(),
        })
    )
    .reset_index()
    .sort_values('Weighted Exposure', ascending=True)
)

fig3 = px.bar(
    group_stats,
    x='Weighted Exposure',
    y='Occupation Group',
    orientation='h',
    color='Weighted Exposure',
    color_continuous_scale='RdYlGn_r',
    hover_data={'Total Employment': ':,', 'Weighted Exposure': ':.3f'},
    title='Employment-Weighted AI Exposure by Occupation Group',
    labels={'Weighted Exposure': 'Avg AI Exposure Score (weighted by employment)'},
)

fig3.add_vline(
    x=weighted_exposure,
    line_dash='dash',
    line_color='black',
    annotation_text=f'MSA avg: {weighted_exposure:.2f}',
    annotation_position='top right',
)

fig3.update_layout(height=600, showlegend=False)
fig3.show()

## 8. Top 20 occupations by exposure-weighted employment

In [10]:
df['Exposure × Employment'] = df['observed_exposure'] * df['Estimated Employment']

top20 = df.nlargest(20, 'Exposure × Employment')[[
    'Occupational Title', 'SOC', 'Estimated Employment',
    'observed_exposure', 'Exposure × Employment', 'Mean Annual'
]]

fig4 = px.bar(
    top20.sort_values('Exposure × Employment'),
    x='Exposure × Employment',
    y='Occupational Title',
    orientation='h',
    color='observed_exposure',
    color_continuous_scale='RdYlGn_r',
    hover_data={
        'SOC': True,
        'Estimated Employment': ':,',
        'observed_exposure': ':.3f',
        'Mean Annual': ':$,.0f',
    },
    title='Top 20 Occupations by Exposure × Employment<br>'
          '<sup>Largest share of AI-exposed labor in the MSA</sup>',
)

fig4.update_layout(height=650, showlegend=False,
                   coloraxis_colorbar_title='Exposure')
fig4.show()

## 9. Visualisation E – Exposed Wage Bill: Employment × Mean Salary × Exposure

In [11]:
df['Exposed Wage Bill'] = df['Estimated Employment'] * df['Mean Annual'] * df['observed_exposure']

total_wage_bill = (df['Estimated Employment'] * df['Mean Annual']).sum()
total_exposed   = df['Exposed Wage Bill'].sum()
print(f"Total wage bill:         ${total_wage_bill:>16,.0f}")
print(f"Exposed wage bill:       ${total_exposed:>16,.0f}  ({total_exposed/total_wage_bill*100:.1f}% of total)")

# --- Group-level rollup ---
group_wage = (
    df.groupby('Occupation Group')
    .agg(
        Exposed_Wage_Bill=('Exposed Wage Bill', 'sum'),
        Total_Employment=('Estimated Employment', 'sum'),
        Weighted_Exposure=('observed_exposure',
                           lambda x: (x * df.loc[x.index, 'Estimated Employment']).sum()
                                     / df.loc[x.index, 'Estimated Employment'].sum()),
    )
    .reset_index()
    .sort_values('Exposed_Wage_Bill', ascending=True)
)

# Readable label for hover
group_wage['Exposed Wage Bill ($B)'] = group_wage['Exposed_Wage_Bill'] / 1e9

fig5 = px.bar(
    group_wage,
    x='Exposed_Wage_Bill',
    y='Occupation Group',
    orientation='h',
    color='Weighted_Exposure',
    color_continuous_scale='RdYlGn_r',
    hover_data={
        'Total_Employment': ':,',
        'Weighted_Exposure': ':.3f',
        'Exposed Wage Bill ($B)': ':.2f',
        'Exposed_Wage_Bill': False,
    },
    title='Exposed Wage Bill by Occupation Group<br>'
          '<sup>Employment × Mean Annual Salary × AI Exposure Score</sup>',
    labels={
        'Exposed_Wage_Bill': 'Exposed Wage Bill ($)',
        'Weighted_Exposure': 'Avg Exposure',
        'Total_Employment': 'Total Employment',
    },
)

fig5.update_xaxes(tickformat='$,.0f')
fig5.update_layout(height=600, showlegend=False,
                   coloraxis_colorbar_title='Avg Exposure')
fig5.show()

# --- Treemap version (occupation-level detail) ---
df_valid = df.dropna(subset=['Mean Annual', 'Exposed Wage Bill'])

fig6 = px.treemap(
    df_valid,
    path=[px.Constant('All Occupations'), 'Occupation Group', 'Occupational Title'],
    values='Exposed Wage Bill',
    color='observed_exposure',
    color_continuous_scale='RdYlGn_r',
    color_continuous_midpoint=0.25,
    hover_data={
        'Estimated Employment': ':,',
        'observed_exposure': ':.3f',
        'Mean Annual': ':$,.0f',
        'Exposed Wage Bill': ':$,.0f',
    },
    title='Exposed Wage Bill Treemap<br>'
          '<sup>Area = Employment × Salary × Exposure | Color = AI Exposure Score (red = higher)</sup>',
)

fig6.update_layout(height=700, coloraxis_colorbar_title='Exposure')
fig6.show()

Total wage bill:         $  16,951,609,960
Exposed wage bill:       $   2,575,965,809  (15.2% of total)


## 10. Summary – Cumulative Exposure: Workers & Salary at Risk

In [12]:
summary_df = df.dropna(subset=['Mean Annual']).copy()
summary_df['Yearly Salary'] = summary_df['Estimated Employment'] * summary_df['Mean Annual']
summary_df['Exposed Yearly Salary'] = summary_df['Yearly Salary'] * summary_df['observed_exposure']

def agg_group(mask, label):
    g = summary_df[mask]
    emp        = g['Estimated Employment'].sum()
    yearly_sal = g['Yearly Salary'].sum()
    exp_sal    = g['Exposed Yearly Salary'].sum()
    avg_salary = yearly_sal / emp if emp else 0
    return {
        'Category':                    label,
        '# Employees':                 emp,
        '% of Workforce':              emp / summary_df['Estimated Employment'].sum() * 100,
        'Avg Annual Salary ($)':       avg_salary,
        'Total Yearly Salary ($M)':    yearly_sal / 1e6,
        'Exposed Yearly Salary ($M)':  exp_sal    / 1e6,
    }

exposed_mask = summary_df['observed_exposure'] > 0
rows = [
    agg_group(exposed_mask,                              'Exposed (score > 0)'),
    agg_group(~exposed_mask,                             'Not Exposed (score = 0)'),
    agg_group(pd.Series(True, index=summary_df.index),  'Total'),
]

tbl = pd.DataFrame(rows)

fig7 = go.Figure(go.Table(
    columnwidth=[220, 120, 130, 180, 195, 210],
    header=dict(
        values=[f'<b>{c}</b>' for c in tbl.columns],
        fill_color='#2c3e50',
        font=dict(color='white', size=13),
        align='center',
        height=36,
    ),
    cells=dict(
        values=[
            tbl['Category'],
            tbl['# Employees'].map('{:,.0f}'.format),
            tbl['% of Workforce'].map('{:.1f}%'.format),
            tbl['Avg Annual Salary ($)'].map('${:,.0f}'.format),
            tbl['Total Yearly Salary ($M)'].map('${:,.1f}M'.format),
            tbl['Exposed Yearly Salary ($M)'].map('${:,.1f}M'.format),
        ],
        fill_color=[['#d5e8d4', '#f8cecc', '#dae8fc']],
        font=dict(size=13),
        align=['left', 'right', 'right', 'right', 'right', 'right'],
        height=32,
    ),
))

fig7.update_layout(
    title='Exposure Summary – Huntsville MSA Labor Market<br>'
          '<sup>Jobs with no exposure score are reported separately; dollar figures in millions</sup>',
    height=260,
    margin=dict(l=20, r=20, t=80, b=20),
)

fig7.show()